# Project B - Context-Gap Distillation

KL between the model's predictions with and without the skill document in context, used
as both the importance signal and the distillation loss.

More session-choppable than Project A: each skill-category adapter is self-contained and
`distill/train.py --resume` skips groups already trained. Budget ~5 GPU-hours.

In [ ]:
import os, subprocess, sys
REPO = '/kaggle/working/myrios'
if not os.path.exists(REPO):
    r = subprocess.run(['git', 'clone', 'https://github.com/USER/myrios.git', REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
    assert r.returncode == 0, 'clone failed - fix the URL in this cell'
os.chdir(REPO)
sys.path.insert(0, REPO)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])
exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu', torch.cuda.get_device_name(0))


RUNS = '/kaggle/working/artifacts/runs_skill'
SCORES = f'{RUNS}/scores_span.jsonl'

In [ ]:
ARCHIVE = '/kaggle/input/myrios-runs-skill/runs_skill.zip'
run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
run(f"python skills/generate_toy_skills.py")


## Gate verification

The riskiest part of the pipeline. High-KL spans must be API identifiers, error codes and
rule clauses; low-KL spans must be markdown scaffolding. Inspect this by eye before
trusting anything trained on it.

In [ ]:
run(f"python kl_gate/score.py --config configs/skill_base.yaml --granularity span --out {SCORES}")
run(f"python kl_gate/inspect_gate.py --scores {SCORES} --top-frac 0.25 --show 6 --out {RUNS}/gate_report.json")


## Distillation and evaluation

`random` is the control that matters: same active-token budget as the KL gate, spans
chosen at random. Any gap between it and `kl_top` is the gate doing real work.

In [ ]:
run(f"python scripts/run_skills.py --config configs/skill_base.yaml --stages distill,eval,report")
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip")


In [ ]:
run(f"python scripts/sweep_kl.py --config configs/skill_base.yaml --fracs 0.1,0.25,0.5 --granularities span,token")
run(f"python eval/skill_report.py --report-dir {RUNS}/report --run-root {RUNS} --out docs/results_skills.md")


In [ ]:
from IPython.display import Image, display
display(Image(f'{RUNS}/report/figures/headline_skills.png'))
display(Image(f'{RUNS}/report/figures/kl_threshold_sweep.png'))
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_skill.zip")
